In [3]:
import os
import re
import warnings
import random
from collections import defaultdict
from typing import Dict, List, Tuple

import queue

import torch
import torch.nn.functional as F
import numpy as np
import numpy as np
import torch
from tqdm.notebook import tqdm
from transformers import GPT2LMHeadModel, GPT2Tokenizer

warnings.filterwarnings("ignore")

/home/Applications/Data/programming/2025/data-portfolio/skils/NN_skils/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
def seed_everything(seed: int):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True

seed_everything(42)

## Задание

1) Реализовать методы `greedy_sampling` и `generate` (1 балл)
2) Реализовать метод `random_sampling` и поддержать его в `generate` (1 балл)
3) Реализовать метод `_beam_search_generate` и поддержать его в `generate` (2 балла)
4) Реализовать методы `apply_top_p`, `apply_top_k`, `apply_temperature` и поддержать их в `generate` (1 балл)  
Все методы необходимо реализовать через векторные операции в torch/numpy везде где это возможно

In [48]:
class Model:
    def __init__(self, model_name: str = "gpt2"):
        self.model = GPT2LMHeadModel.from_pretrained(model_name)
        self.tokenizer = GPT2Tokenizer.from_pretrained(model_name)
        self.tokenizer.pad_token = self.tokenizer.eos_token
        self.vocab_size = self.tokenizer.vocab_size

    def greedy_sampling(self, logits: torch.Tensor) -> int:
        if len(logits) != self.vocab_size:
            raise AttributeError(f"Logits size = {logits.shape}. Vocab size = (1, {self.vocab_size})")
        if isinstance(logits, torch.Tensor):
            return torch.argmax(logits).item()
        raise AttributeError("Logits must be a torch.Tensor")
    

    def random_sampling(self, logits: torch.Tensor) -> int:
        if len(logits) != self.vocab_size:
            raise AttributeError(f"Logits size = {logits.shape}. Vocab size = (1, {self.vocab_size})")
        if isinstance(logits, torch.Tensor):
            probs = torch.softmax(logits, dim=0)
            return torch.multinomial(probs, num_samples=1).item()
        else:
            raise AttributeError("Logits must be a torch.Tensor")        
    
    def _beam_search_generate(
        self,
        prompt: str,
        max_length: int,
        num_beams: int
    ) -> str:
            
        device = self.model.device
        input_tokens = self.tokenizer(prompt, return_tensors="pt")
        input_tokens = {k: v.to(device) for k, v in input_tokens.items()}
        
        beams = []
        beams.append((input_tokens["input_ids"], 0.0))
        
        self.model.eval()
        with torch.no_grad():
            for i in range(max_length):
                candidates = []
                for input_ids, score in beams:
                    
                    outputs = self.model(input_ids=input_ids)
                    logits = outputs.logits[0, -1, :]
                    
                    log_probs = torch.log_softmax(logits, dim=-1)
                    
                    values, indices = torch.topk(log_probs, k=num_beams)
                    for v, idx in zip(values, indices):
                        new_inputs_ids = torch.cat(
                            (input_ids, idx.unsqueeze(0).unsqueeze(0)), dim=1
                        )
                        candidates.append((new_inputs_ids, score + v.item()))
                sorted_candidates = sorted(candidates, key=lambda x: x[1], reverse=True)
                beams = sorted_candidates[:num_beams]
                
                if all(b[0][0, -1].item() == self.tokenizer.eos_token_id for b in beams):
                    break
                
        best = max(beams, key=lambda x: x[1])[0]
        generated_part = best[0, input_tokens["input_ids"].shape[1]:]
        result = self.tokenizer.decode(generated_part, skip_special_tokens=False)
        return result
                        
    def apply_temperature(self, logits: torch.Tensor, temperature: float = 1.0) -> torch.Tensor:
        if len(logits) != self.vocab_size:
            raise AttributeError(f"Logits size = {logits.shape}. Vocab size = (1, {self.vocab_size})")
        if isinstance(logits, torch.Tensor):
            if temperature <= 0:
                raise ValueError("Temperature must be positive and non-zero.")
            logits /= temperature
        else:
            raise AttributeError("Logits must be a torch.Tensor")  

    def _apply_top_p(self, logits: torch.Tensor, top_p: float = 1.0) -> torch.Tensor:
        if len(logits) != self.vocab_size:
            raise AttributeError(f"Logits size = {logits.shape}. Vocab size = (1, {self.vocab_size})")
        if isinstance(logits, torch.Tensor):
            if top_p < 1.0:
                prob = torch.softmax(logits, dim=0)
                sorted, indices = torch.sort(prob, descending=True)
                cumsum = torch.cumsum(sorted, dim=0)
                mask_indices = indices[cumsum <= top_p]  
                if len(mask_indices) == 0:
                    mask_indices = indices[:1]  
                device = indices.device
                mask = torch.zeros_like(prob, dtype=bool, device=device)
                mask[mask_indices] = True
                logits[torch.logical_not(mask)] = -float('inf')
        else:
            raise AttributeError("Logits must be a torch.Tensor")       
        

    def _apply_top_k(self, logits: torch.Tensor, top_k: float) -> torch.Tensor:
        if len(logits) != self.vocab_size:
            raise AttributeError(f"Logits size = {logits.shape}. Vocab size = (1, {self.vocab_size})")
        if isinstance(logits, torch.Tensor):
            if top_k > 0:
                _, indices = torch.topk(logits, top_k)
                device = indices.device
                mask = torch.zeros_like(logits, dtype=bool, device=device)
                mask[indices] = True
                logits[torch.logical_not(mask)] = -float('inf')    
        else:
            raise AttributeError("Logits must be a torch.Tensor")  
    
    def generate(
        self,
        prompt: str,
        max_length: int = 50,
        strategy: str = "greedy",
        temperature: float = 1.0,
        top_k: int = 0,
        top_p: float = 1.0,
        num_beams: int = 3
    ) -> str:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.model = self.model.to(device)
        self.model.eval()
        self.model.use_cache = True
        
        result = ""
        
        with torch.no_grad():
            if strategy == "beam":
                return self._beam_search_generate(prompt, max_length, num_beams)
            elif strategy == "greedy":
                inputs: dict = self.tokenizer(prompt, return_tensors="pt")
                inputs = {k: v.to(device) for k, v in inputs.items()}
                for _ in range(max_length):
                    outputs = self.model(**inputs)
                    logits = outputs.logits[0, -1, :].clone()
                    
                    self.apply_temperature(logits, temperature)
                    self._apply_top_k(logits, top_k)
                    self._apply_top_p(logits, top_p)
                    
                    next_token = self.greedy_sampling(logits)
                    result += self.tokenizer.decode(next_token)
                    if next_token == self.tokenizer.eos_token_id:
                        break
                    inputs["input_ids"] = torch.cat((inputs["input_ids"], torch.tensor([[next_token]], device=device)), dim=1)
                    inputs["attention_mask"] = torch.cat((inputs["attention_mask"], torch.tensor([[1]], device=device)), dim=1)
            elif strategy == "multinomial":
                inputs: dict = self.tokenizer(prompt, return_tensors="pt")
                inputs = {k: v.to(device) for k, v in inputs.items()}
                for _ in range(max_length):
                    outputs = self.model(**inputs)
                    logits = outputs.logits[0, -1, :].clone()
                    
                    self.apply_temperature(logits, temperature)
                    self._apply_top_k(logits, top_k)
                    self._apply_top_p(logits, top_p)
                    
                    next_token = self.random_sampling(logits)
                    result += self.tokenizer.decode(next_token)
                    if next_token == self.tokenizer.eos_token_id:
                        break
                    inputs["input_ids"] = torch.cat((inputs["input_ids"], torch.tensor([[next_token]], device=device)), dim=1)
                    inputs["attention_mask"] = torch.cat((inputs["attention_mask"], torch.tensor([[1]], device=device)), dim=1)
        return result

In [49]:
# Продемонстрируйте результат работы `generate` при различных параметрах
obj = Model()
result_text_multinomial = obj.generate("Generate one sentence. Today was a wonderful day and I went to play football with", strategy="multinomial", temperature=0.9, top_k=20, top_p=0.9)
result_text_greedy = obj.generate("Generate one sentence. Today was a wonderful day and I went to play football with", strategy="greedy", top_k=20, top_p=0.9)
result_text_beem = obj.generate("Generate one sentence. Today was a wonderful day and I went to play football with", strategy="beam")

In [50]:
print(f"Random generate: {result_text_multinomial}")
print(f"Greedy sampling: {result_text_greedy}")
print(f"Beem sampling: {result_text_beem}")

Random generate:  a friend, who is now my girlfriend. I was so happy. We played for the first time since I left for college. I was so happy.

"But I'm not happy. I'm not happy with that. I'm not
Greedy sampling:  my family. I was so happy. I was so happy. I was so happy. I was so happy. I was so happy. I was so happy. I was so happy. I was so happy. I was so happy. I was
Beem sampling:  my family. I was so happy to be back. I'm so happy to be back. I'm so happy to be back. I'm so happy to be back. I'm so happy to be back. I'm so happy to be back


In [51]:
# Продемонстрируйте результат работы `generate` при различных параметрах
result_text_multinomial = obj.generate("Beautiful girl", strategy="multinomial", temperature=1.5, top_k=20, top_p=0.9)
result_text_greedy = obj.generate("Beautiful girl", strategy="greedy", top_k=20, top_p=0.9)
result_text_beem = obj.generate("Beautiful girl", strategy="beam")

In [52]:
print(f"Random generate: {result_text_multinomial}")
print(f"Greedy sampling: {result_text_greedy}")
print(f"Beem sampling: {result_text_beem}")

Random generate: . This is the best. The best girl you ever had, and this isn't even the first time you've had her, and the only other time she'd done this in a million years is on your wedding day."

"And she
Greedy sampling: , I'm so happy to be here. I'm so happy to be here. I'm so happy to be here. I'm so happy to be here. I'm so happy to be here. I'm so happy to be here. I
Beem sampling: . I love her. I love her. I love her. I love her. I love her. I love her. I love her. I love her. I love her. I love her. I love her. I love her. I


In [ ]:
from unittest.mock import patch

def test_greedy_sampling():
    with patch.object(Model, "__init__", lambda self:  None):
        obj = Model()
        obj.vocab_size = 5
        assert obj.greedy_sampling(torch.Tensor([0, 1, 2, 3, 4])) == 4
        assert obj.greedy_sampling(torch.Tensor([0, 0, 0, 0, 0])) == 0
        assert obj.greedy_sampling(torch.Tensor([0, -1, 1, 0.5, 0.001])) == 2
        assert obj.greedy_sampling(torch.Tensor([0, -1e-3, -1e-3, -1e-3, 1e-4])) == 4
test_greedy_sampling()

def test_apply_top_k():
    
    with patch.object(Model, "__init__", lambda self: None):
        
        obj = Model()
        obj.vocab_size = 5
        inf = float('inf')
        
        logits = torch.Tensor([0, 1, 2, 3, 4])
        obj._apply_top_k(logits, 2)
        assert all(logits == torch.Tensor([-inf, -inf, -inf, 3, 4]))
        
        logits = torch.Tensor([-2, 0, -1, -5, 0])
        obj._apply_top_k(logits, 2)
        assert all(logits == torch.Tensor([-inf, 0, -inf, -inf, 0]))
        
        logits = torch.Tensor([1, 0, 3, 0, 0])
        obj._apply_top_k(logits, 2)
        assert all(logits == torch.Tensor([1, -inf, 3, -inf, -inf]))

test_apply_top_k()

def test_apply_top_p():
    
    with patch.object(Model, "__init__", lambda self: None):
        
        obj = Model()
        obj.vocab_size = 5
        inf = float('inf')
        
        logits = torch.Tensor([0, 1, 2, 3, 4])
        obj._apply_top_p(logits, 0.99)
        assert all(logits == torch.Tensor([-inf, 1, 2, 3, 4]))
        
        logits = torch.Tensor([0, 1, 2, 3, 4])
        obj._apply_top_p(logits, 0.64)
        assert all(logits == torch.Tensor([-inf, -inf, -inf, -inf, 4]))
        
        logits = torch.Tensor([1, 0, 3, 0, 2])
        obj._apply_top_p(logits, 0.86)
        assert all(logits == torch.Tensor([-inf, -inf, 3, -inf, 2]))

test_apply_top_p()

def test_temperature():
    with patch.object(Model, "__init__", lambda self: None):
        obj = Model()
        obj.vocab_size = 5
        
        logits = torch.Tensor([1, 2, 3, 4, 5])
        obj.apply_temperature(logits, 1)
        assert all(logits == torch.Tensor([1, 2, 3, 4, 5]))
        
test_temperature()